In [5]:
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal
import matplotlib.pyplot as plt

# Define output directory for spectrogram files and create it if not exists
output_dir = 'spectrogram_output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Function to load audio and annotations from files
def load_audio_and_annotation(file_base_name):
    audio_path = file_base_name + '.wav'
    sample_rate, samples = wavfile.read(audio_path)
    
    annot_file_path = file_base_name + '.Table.1.selections.txt'
    df = pd.read_csv(annot_file_path, sep='\t')
    
    return sample_rate, samples, df

# Function to extract audio segments based on annotation time frames
def extract_audio_segments(samples, sample_rate, annotations):
    segments = []
    
    # Iterate through each row of the annotation DataFrame and extract the audio
    for _, row in annotations.iterrows():
        start_time = row['Begin Time (s)']
        end_time = row['End Time (s)']
        
        start_sample = int(start_time * sample_rate)
        end_sample = int(end_time * sample_rate)
        
        segment = samples[start_sample:end_sample]
        
        segments.append({
            'Selection': row['Selection'],
            'View': row['View'],
            'Channel': row['Channel'],
            'Begin Time (s)': start_time,
            'End Time (s)': end_time,
            'Low Freq (Hz)': row['Low Freq (Hz)'],
            'High Freq (Hz)': row['High Freq (Hz)'],
            'Delta Time (s)': row['Delta Time (s)'],
            'Delta Freq (Hz)': row['Delta Freq (Hz)'],
            'Avg Power Density (dB FS/Hz)': row['Avg Power Density (dB FS/Hz)'],
            'Annotation': row['Annotation'],
            'audio_segment': segment
        })
        
    return segments

def generate_spectrogram(audio_segment, sample_rate, fmin=20, fmax=1000, nperseg=2456, nfft=4096, noverlap=1228):
    """
    Generate spectrogram for an audio segment with frequency slicing.

    Parameters:
    - audio_segment: The raw audio data (1D array).
    - sample_rate: The sample rate of the audio data.
    - fmin: The minimum frequency for spectrogram (default: 20 Hz).
    - fmax: The maximum frequency for spectrogram (default: 1000 Hz).
    - nperseg, nfft, noverlap: Parameters for spectrogram calculation.
    
    Returns:
    - frequencies: Frequency bins for the spectrogram after trimming to fmin-fmax.
    - times: Time bins for the spectrogram.
    - spectrogram: The 2D spectrogram data array after trimming.
    """
    
    # Ensure that nperseg is not greater than the audio segment length
    nperseg = min(nperseg, len(audio_segment))  # Make sure nperseg does not exceed audio segment length
    
    # Adjust nfft to be at least as large as nperseg
    nfft = max(nperseg, nfft)  # Ensure that nfft is >= nperseg
    
    # Adjust noverlap if it exceeds nperseg, ensuring it's at most nperseg / 2
    if noverlap >= nperseg:
        print(f"Warning: noverlap ({noverlap}) is >= nperseg ({nperseg}), adjusting noverlap.")
        noverlap = nperseg // 2  # Adjust noverlap to half of nperseg to prevent errors
    
    # Generate the spectrogram using scipy.signal.spectrogram
    frequencies, times, spectrogram = signal.spectrogram(audio_segment, sample_rate, 
                                                         nperseg=nperseg, nfft=nfft, 
                                                         noverlap=noverlap, window='hann')
    
    # Trim tiny values to avoid noise issues when plotting on a log scale
    spectrogram[spectrogram < 0.001] = 0.001
    
    # Slice the frequency range from fmin to fmax
    freq_slice = np.where((frequencies >= fmin) & (frequencies <= fmax))
    
    # Keep only the relevant frequencies and the corresponding spectrogram
    frequencies = frequencies[freq_slice]
    spectrogram = spectrogram[freq_slice, :]
    
    return frequencies, times, spectrogram


# Function to save spectrogram as image
def save_spectrogram(filename, spectrogram_data, times, frequencies):
    plt.figure(figsize=(10, 6))
    plt.pcolormesh(times, frequencies, 10 * np.log10(spectrogram_data), shading='auto')
    plt.colorbar(label='Log Power (dB)')
    plt.title('Spectrogram')
    plt.xlabel('Time [s]')
    plt.ylabel('Frequency [Hz]')
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

# Function to pre-process and find the longest call (in time and frequency range)
def get_baseline_specs(final_df, sample_rate):
    max_duration = 0
    max_freq_range = 0
    longest_call_segment = None
    
    for _, segment_info in final_df.iterrows():
        start_time = segment_info['Begin Time (s)']
        end_time = segment_info['End Time (s)']
        duration = end_time - start_time
        freq_range = segment_info['High Freq (Hz)'] - segment_info['Low Freq (Hz)']
        
        if duration > max_duration:
            max_duration = duration
            longest_call_segment = segment_info
            
        if freq_range > max_freq_range:
            max_freq_range = freq_range
    
    # Set the baseline spectrogram parameters
    fmin = longest_call_segment['Low Freq (Hz)']
    fmax = longest_call_segment['High Freq (Hz)']
    
    # Determine nperseg, nfft, and noverlap based on the longest call duration
    duration_samples = int(max_duration * sample_rate)
    nperseg = duration_samples  # We'll use the longest duration for nperseg
    nfft = 4096  # Fixed, can adjust depending on use case
    noverlap = nperseg // 2  # Default overlap half of segment
    
    return fmin, fmax, nperseg, nfft, noverlap

# Define folder paths
folder_paths = [
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Guttural rupe',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Rupes A and B',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Moan'
]

# Initialize list to collect DataFrames for all segments
dataset = []

# Process each folder and file for annotations and audio
for folder_path in folder_paths:
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_base_name = os.path.splitext(file)[0]
            file_base_path = os.path.join(folder_path, file_base_name)
            
            sample_rate, samples, df = load_audio_and_annotation(file_base_path)
            
            segments = extract_audio_segments(samples, sample_rate, df)
            
            df_segments = pd.DataFrame(segments)
            
            dataset.append(df_segments)

# Combine all segment DataFrames into one final dataset
final_df = pd.concat(dataset, ignore_index=True)

# Determine baseline spectrogram specifications (maximum call duration and frequency range)
fmin, fmax, nperseg, nfft, noverlap = get_baseline_specs(final_df, sample_rate)

# Process and save spectrograms for each annotated call and the "no-call" periods
for index, segment_info in final_df.iterrows():
    audio_segment = segment_info['audio_segment']
    start_time = segment_info['Begin Time (s)']
    end_time = segment_info['End Time (s)']
    
    low_freq = segment_info['Low Freq (Hz)']
    high_freq = segment_info['High Freq (Hz)']
    
    frequencies, times, spectrogram_data = generate_spectrogram(audio_segment, sample_rate, 
                                                               fmin=low_freq, fmax=high_freq,
                                                               nperseg=nperseg, nfft=nfft, noverlap=noverlap)
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    metadata = {
        'Selection': segment_info['Selection'],
        'Begin Time (s)': start_time,
        'End Time (s)': end_time,
        'Annotation': segment_info['Annotation'],
        'Central Time (s)': (start_time + end_time) / 2,  # Calculate the central time of the call
        'Spectrogram Shape': spectrogram_data.shape
    }
    
    spectrogram_filename = f"{output_dir}/{segment_info['Selection']}_{start_time}-{end_time}_spectrogram.npz"
    
    np.savez(spectrogram_filename, spectrogram=spectrogram_data, metadata=metadata)

# Handle "no-call" segments (gaps)
sorted_segments = final_df.sort_values(by='Begin Time (s)')

for i in range(1, len(sorted_segments)):
    prev_end_time = sorted_segments.iloc[i - 1]['End Time (s)']
    curr_start_time = sorted_segments.iloc[i]['Begin Time (s)']
    
    if curr_start_time > prev_end_time:
        start_sample = int(prev_end_time * sample_rate)
        end_sample = int(curr_start_time * sample_rate)
        no_call_segment = samples[start_sample:end_sample]
        
        frequencies, times, spectrogram_data = generate_spectrogram(no_call_segment, sample_rate, 
                                                                   fmin=fmin, fmax=fmax,
                                                                   nperseg=nperseg, nfft=nfft, noverlap=noverlap)
        
        no_call_filename = f"{output_dir}/no_call_{prev_end_time}-{curr_start_time}_spectrogram.npz"
        
        metadata = {
            'Selection': 'no-call',
            'Begin Time (s)': prev_end_time,
            'End Time (s)': curr_start_time,
            'Spectrogram Shape': spectrogram_data.shape
        }
        
        np.savez(no_call_filename, spectrogram=spectrogram_data, metadata=metadata)
